# FoundationStereo KITTI fine-tuning and cross-dataset evaluation

This notebook runs two supervised fine-tuning protocols using the official FoundationStereo checkpoint:

**Protocol A**

- Train: all KITTI 2012 training pairs
- Validation: 40 KITTI 2015 pairs
- Test: held-out 160 KITTI 2015 pairs

**Protocol B**

- Train: 155 KITTI 2012 pairs
- Validation: 39 KITTI 2012 pairs
- Test: all 200 KITTI 2015 training pairs

For each protocol it compares the original zero-shot model with the fine-tuned model on KITTI and Middlebury. The DINOv2/DepthAnything feature extractor remains frozen; the stereo matching and refinement network is fine-tuned.

Upload this notebook and `foundationstereo_source_bundle.zip` to Colab, enable a GPU, and run cells in order.


In [ ]:
# Colab setup
!pip -q install omegaconf open3d imageio opencv-python einops timm scipy scikit-image gdown ruamel.yaml transformations trimesh joblib pandas

from pathlib import Path
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import shutil
import subprocess
import sys
import time
import zipfile

import numpy as np
import torch

BUNDLE_ZIP = Path("/content/foundationstereo_source_bundle.zip")
if not BUNDLE_ZIP.exists():
    raise FileNotFoundError("Upload foundationstereo_source_bundle.zip before running this cell.")

BUNDLE = Path("/content/foundationstereo_bundle")
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
BUNDLE.mkdir()
with zipfile.ZipFile(BUNDLE_ZIP) as archive:
    archive.extractall(BUNDLE)

SOURCE_ROOT = next(
    (p.parent.parent for p in BUNDLE.rglob("core/foundation_stereo.py")),
    None,
)
if SOURCE_ROOT is None:
    raise FileNotFoundError("The source bundle does not contain core/foundation_stereo.py.")
sys.path.insert(0, str(SOURCE_ROOT))

ROOT = Path("/content/foundationstereo_kitti")
DATA = ROOT / "data"
CHECKPOINT = ROOT / "checkpoints" / "23-51-11"
OUTPUT = ROOT / "outputs"
DATA.mkdir(parents=True, exist_ok=True)
CHECKPOINT.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

print("Source root:", SOURCE_ROOT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Colab GPU runtime before continuing.")


In [ ]:
# Download and extract KITTI 2012/2015 training data
import requests

def download(url, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size > 0:
        print("Already downloaded:", path.name)
        return
    print("Downloading:", path.name)
    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        stream=True,
        timeout=120,
    )
    response.raise_for_status()
    with path.open("wb") as output:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                output.write(chunk)

def extract_once(archive_path, destination):
    destination = Path(destination)
    marker = destination / ".extracted"
    if marker.exists():
        return
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(destination)
    marker.write_text("ok")

KITTI12 = DATA / "kitti2012"
KITTI15 = DATA / "kitti2015"
download(
    "https://s3.eu-central-1.amazonaws.com/avg-kitti/data_stereo_flow.zip",
    DATA / "data_stereo_flow.zip",
)
download(
    "https://s3.eu-central-1.amazonaws.com/avg-kitti/data_scene_flow.zip",
    DATA / "data_scene_flow.zip",
)
extract_once(DATA / "data_stereo_flow.zip", KITTI12)
extract_once(DATA / "data_scene_flow.zip", KITTI15)

def find_training_root(root, dataset):
    root = Path(root)
    image_folder = "colored_0" if dataset == "kitti2012" else "image_2"
    candidates = [
        path.parent.parent
        for path in root.rglob(image_folder)
        if path.is_dir() and path.parent.name == "training"
    ]
    if not candidates:
        archive_names = []
        for archive_path in DATA.glob("*.zip"):
            try:
                with zipfile.ZipFile(archive_path) as archive:
                    archive_names.extend(archive.namelist()[:20])
            except zipfile.BadZipFile:
                archive_names.append(f"{archive_path.name}: invalid ZIP")
        raise FileNotFoundError(
            f"Could not find a training/image_2 directory under {root}. "
            f"Extracted entries: {list(root.rglob('*'))[:20]}. "
            f"Archive entries: {archive_names}"
        )
    return candidates[0]

KITTI12_ROOT = find_training_root(KITTI12, "kitti2012")
KITTI15_ROOT = find_training_root(KITTI15, "kitti2015")
print("KITTI 2012:", KITTI12_ROOT)
print("KITTI 2015:", KITTI15_ROOT)


In [ ]:
# Download the official FoundationStereo checkpoint
from huggingface_hub import hf_hub_download

MODEL_FILE = CHECKPOINT / "model_best_bp2.pth"
CFG_FILE = CHECKPOINT / "cfg.yaml"
HF_REPO = "steve-redefine/FoundationStereoWeights"

if not CFG_FILE.exists() or CFG_FILE.stat().st_size < 100:
    print("Downloading cfg.yaml from Hugging Face")
    hf_hub_download(
        repo_id=HF_REPO,
        repo_type="dataset",
        filename="23-51-11/cfg.yaml",
        local_dir=CHECKPOINT.parent,
        local_dir_use_symlinks=False,
    )

if not MODEL_FILE.exists() or MODEL_FILE.stat().st_size < 1_000_000_000:
    print("Downloading model_best_bp2.pth from Hugging Face (about 3.3 GiB)")
    hf_hub_download(
        repo_id=HF_REPO,
        repo_type="dataset",
        filename="23-51-11/model_best_bp2.pth",
        local_dir=CHECKPOINT.parent,
        local_dir_use_symlinks=False,
    )

if not MODEL_FILE.exists() or MODEL_FILE.stat().st_size < 1_000_000_000:
    raise FileNotFoundError(
        "The FoundationStereo checkpoint was not downloaded completely. "
        "Delete the incomplete file and rerun this cell."
    )

print("Checkpoint:", MODEL_FILE)
print("Size GiB:", round(MODEL_FILE.stat().st_size / 1024**3, 3))


In [ ]:
# Dataset indexing, deterministic splits, and stereo-safe crops
from dataclasses import dataclass
import cv2
import imageio.v2 as imageio

@dataclass(frozen=True)
class KITTISample:
    dataset: str
    index: str
    root: Path

    @property
    def image_folder(self):
        return "colored_0" if self.dataset == "kitti2012" else "image_2"

    @property
    def right_folder(self):
        return "colored_1" if self.dataset == "kitti2012" else "image_3"

    def frame_name(self, folder):
        directory = self.root / "training" / folder
        if self.dataset == "kitti2012":
            suffixes = ["_10", ""]
            candidates = [directory / f"{self.index}{suffix}.png" for suffix in suffixes]
        else:
            candidates = [directory / f"{self.index}{suffix}.png" for suffix in ["_10", ""]]
            candidates.extend(sorted(directory.glob(f"{self.index}_*.png")))
        for candidate in candidates:
            if candidate.exists():
                return candidate.name
        raise FileNotFoundError(
            f"No frame found for {self.dataset}/{self.index} in {directory}"
        )

    @property
    def left(self):
        return self.root / "training" / self.image_folder / self.frame_name(self.image_folder)

    @property
    def right(self):
        return self.root / "training" / self.right_folder / self.frame_name(self.right_folder)

    @property
    def disp_occ(self):
        folder = "disp_occ_0" if self.dataset == "kitti2015" else "disp_occ"
        return self.root / "training" / folder / self.frame_name(folder)

    @property
    def disp_noc(self):
        folder = "disp_noc_0" if self.dataset == "kitti2015" else "disp_noc"
        return self.root / "training" / folder / self.frame_name(folder)

def index_kitti(dataset, root):
    image_folder = "colored_0" if dataset == "kitti2012" else "image_2"
    image_root = Path(root) / "training" / image_folder
    image_files = sorted(image_root.glob("*.png"))
    indices = sorted({
        p.stem.split("_", 1)[0]
        for p in image_files
    })
    samples = [KITTISample(dataset, index, Path(root)) for index in indices]
    for sample in samples:
        for path in [sample.left, sample.right, sample.disp_occ, sample.disp_noc]:
            if not path.exists():
                raise FileNotFoundError(path)
    return samples

kitti12_samples = index_kitti("kitti2012", KITTI12_ROOT)
kitti15_samples = index_kitti("kitti2015", KITTI15_ROOT)
print("KITTI 2012 pairs:", len(kitti12_samples))
print("KITTI 2015 pairs:", len(kitti15_samples))
if len(kitti12_samples) != 194 or len(kitti15_samples) != 200:
    raise RuntimeError("Unexpected KITTI training-pair count.")

rng = np.random.default_rng(2025)
k15_order = rng.permutation(len(kitti15_samples))
k12_order = rng.permutation(len(kitti12_samples))

protocol_a = {
    "train": kitti12_samples,
    "val": [kitti15_samples[i] for i in k15_order[:40]],
    "test": [kitti15_samples[i] for i in k15_order[40:]],
}
protocol_b = {
    "train": [kitti12_samples[i] for i in k12_order[:155]],
    "val": [kitti12_samples[i] for i in k12_order[155:]],
    "test": kitti15_samples,
}

for name, split in [("A", protocol_a), ("B", protocol_b)]:
    print(f"Protocol {name}: train={len(split['train'])}, val={len(split['val'])}, test={len(split['test'])}")

def load_kitti(sample):
    left = imageio.imread(sample.left)
    right = imageio.imread(sample.right)
    occ = imageio.imread(sample.disp_occ).astype(np.float32) / 256.0
    noc = imageio.imread(sample.disp_noc).astype(np.float32) / 256.0
    return left, right, occ, noc

def random_crop(left, right, disp, height=256, width=512, rng=None):
    rng = rng or np.random.default_rng()
    h, w = left.shape[:2]
    if h < height or w < width:
        raise ValueError(f"Image {left.shape[:2]} is smaller than crop {(height, width)}")
    y = int(rng.integers(0, h - height + 1))
    x = int(rng.integers(0, w - width + 1))
    return (
        left[y:y+height, x:x+width],
        right[y:y+height, x:x+width],
        disp[y:y+height, x:x+width],
    )


In [ ]:
# Model loading, supervised fine-tuning, and evaluation
from omegaconf import OmegaConf
from core.foundation_stereo import FoundationStereo
from core.utils.utils import InputPadder

def make_model():
    cfg = OmegaConf.load(CHECKPOINT / "cfg.yaml")
    cfg["vit_size"] = "vitl"
    cfg["mixed_precision"] = True
    cfg["valid_iters"] = 32
    model = FoundationStereo(cfg)
    state = torch.load(
        CHECKPOINT / "model_best_bp2.pth",
        map_location="cpu",
        weights_only=False,
    )
    model.load_state_dict(state["model"])
    model.cuda()
    model.eval()
    for parameter in model.feature.dino.parameters():
        parameter.requires_grad = False
    return model

def set_backbone_eval(model):
    model.feature.dino.eval()
    for parameter in model.feature.dino.parameters():
        parameter.requires_grad = False

def make_optimizer(model, lr=1e-5):
    trainable = [p for p in model.parameters() if p.requires_grad]
    return torch.optim.AdamW(trainable, lr=lr, weight_decay=1e-5)

def disparity_loss(predictions, target, valid):
    weights = []
    gamma = 0.9
    for index in range(len(predictions)):
        weights.append(gamma ** (len(predictions) - index - 1))
    weights = np.asarray(weights, dtype=np.float32)
    weights = weights / weights.sum()
    loss = target.new_zeros(())
    for weight, prediction in zip(weights, predictions):
        prediction = prediction[:, 0]
        error = torch.abs(prediction - target)
        smooth = torch.where(error < 1.0, 0.5 * error.square(), error - 0.5)
        loss = loss + float(weight) * smooth[valid].mean()
    return loss

def batch_from_sample(sample, crop_rng):
    left, right, occ, _ = load_kitti(sample)
    left, right, occ = random_crop(left, right, occ, rng=crop_rng)
    x0 = torch.from_numpy(left).cuda().float().permute(2, 0, 1)[None]
    x1 = torch.from_numpy(right).cuda().float().permute(2, 0, 1)[None]
    target = torch.from_numpy(occ).cuda().float()[None]
    valid = torch.isfinite(target) & (target > 0)
    return x0, x1, target, valid

def train_model(samples, val_samples, output_path, steps=5000, lr=1e-5, seed=2025, eval_interval=50, scheduler_patience=2, early_stopping_patience=3):
    torch.cuda.empty_cache()
    model = make_model()
    optimizer = make_optimizer(model, lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=scheduler_patience,
        threshold=1e-3, min_lr=1e-7,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=True)
    crop_rng = np.random.default_rng(seed)
    model.train()
    set_backbone_eval(model)
    losses = []
    best_d1 = float("inf")
    stale_evals = 0
    for step in range(steps):
        sample = samples[int(crop_rng.integers(0, len(samples)))]
        x0, x1, target, valid = batch_from_sample(sample, crop_rng)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(True):
            _, predictions = model.forward(x0, x1, iters=12, test_mode=False)
            loss = disparity_loss(predictions, target, valid)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(optimizer.param_groups[0]["params"], 1.0)
        scaler.step(optimizer)
        scaler.update()
        losses.append(float(loss.detach().cpu()))
        if (step + 1) % 100 == 0:
            print(f"step {step + 1}/{steps}, loss {np.mean(losses[-100:]):.5f}", flush=True)
        if (step + 1) % eval_interval == 0 or step + 1 == steps:
            val_rows = kitti_metrics(model, val_samples)
            val_noc = [row for row in val_rows if row["mask"] == "noc"]
            total_valid = sum(row["valid_pixels"] for row in val_noc)
            total_bad = sum(row["bad_pixels"] for row in val_noc)
            val_d1 = 100.0 * total_bad / total_valid
            scheduler.step(val_d1)
            current_lr = optimizer.param_groups[0]["lr"]
            print(f"validation step {step + 1}: D1-noc {val_d1:.4f}%, lr {current_lr:.2e}", flush=True)
            if val_d1 < best_d1:
                best_d1 = val_d1
                output_path.parent.mkdir(parents=True, exist_ok=True)
                torch.save({"model": model.state_dict(), "steps": step + 1, "val_d1_noc": best_d1}, output_path)
                stale_evals = 0
            else:
                stale_evals += 1
                if stale_evals >= early_stopping_patience:
                    print(f"early stopping at step {step + 1}; no validation improvement for {stale_evals} evaluations", flush=True)
                    break
            model.train()
            set_backbone_eval(model)
    state = torch.load(output_path, map_location="cpu", weights_only=False)
    model.load_state_dict(state["model"])
    model.eval()
    set_backbone_eval(model)
    return model

def predict(model, sample, iters=32):
    left = imageio.imread(sample.left)
    right = imageio.imread(sample.right)
    h, w = left.shape[:2]
    x0 = torch.from_numpy(left).cuda().float().permute(2, 0, 1)[None]
    x1 = torch.from_numpy(right).cuda().float().permute(2, 0, 1)[None]
    padder = InputPadder(x0.shape, divis_by=32, force_square=False)
    x0, x1 = padder.pad(x0, x1)
    with torch.no_grad(), torch.cuda.amp.autocast(True):
        disp = model.forward(x0, x1, iters=iters, test_mode=True)
    return padder.unpad(disp.float()).detach().cpu().numpy().reshape(h, w)

def kitti_metrics(model, samples):
    rows = []
    model.eval()
    set_backbone_eval(model)
    total_samples = len(samples)
    print(f"Starting KITTI inference on {total_samples} samples...", flush=True)
    for sample_number, sample in enumerate(samples, start=1):
        if sample_number == 1 or sample_number % 10 == 0 or sample_number == total_samples:
            print(
                f"Inference {sample_number}/{total_samples} "
                f"({sample.dataset}/{sample.index})",
                flush=True,
            )
        prediction = predict(model, sample)
        _, _, occ, noc = load_kitti(sample)
        for name, gt in [("all", occ), ("noc", noc)]:
            valid = np.isfinite(prediction) & np.isfinite(gt) & (gt > 0)
            error = np.abs(prediction[valid] - gt[valid])
            d1 = (error > 3.0) & (error / np.maximum(gt[valid], 1e-6) > 0.05)
            rows.append({
                "dataset": sample.dataset,
                "index": sample.index,
                "mask": name,
                "valid_pixels": int(valid.sum()),
                "epe": float(error.mean()),
                "bad_pixels": int(d1.sum()),
                "d1": float(d1.mean() * 100.0),
            })
    return rows

def summarize(rows, label):
    print(f"\n{label}")
    for mask in ["all", "noc"]:
        subset = [row for row in rows if row["mask"] == mask]
        total_valid = sum(row["valid_pixels"] for row in subset)
        total_bad = sum(row["bad_pixels"] for row in subset)
        weighted_epe = sum(
            row["epe"] * row["valid_pixels"] for row in subset
        ) / total_valid
        global_d1 = 100.0 * total_bad / total_valid
        mean_image_d1 = np.mean([row["d1"] for row in subset])
        print(
            mask,
            "weighted EPE", f"{weighted_epe:.4f}",
            "global D1", f"{global_d1:.4f}%",
            "mean-image D1", f"{mean_image_d1:.4f}%",
        )


In [ ]:
# Zero-shot baseline on the complete KITTI 2012 and KITTI 2015 training sets
import json
zero_shot_model = make_model()
zero_shot_kitti12 = kitti_metrics(zero_shot_model, kitti12_samples)
zero_shot_kitti15 = kitti_metrics(zero_shot_model, kitti15_samples)
summarize(zero_shot_kitti12, "Zero-shot KITTI 2012 (all 194 pairs)")
summarize(zero_shot_kitti15, "Zero-shot KITTI 2015 (all 200 pairs)")
zero_shot_results = {
    "kitti2012": zero_shot_kitti12,
    "kitti2015": zero_shot_kitti15,
}
with (OUTPUT / "zero_shot_kitti.json").open("w") as file:
    json.dump(zero_shot_results, file, indent=2)
del zero_shot_model
torch.cuda.empty_cache()


### Expected reference results

The FoundationStereo paper reports the following zero-shot KITTI D1 values using the official pretrained model:

| Dataset | Paper zero-shot D1 |
|---|---:|
| KITTI 2012 | 2.3% |
| KITTI 2015 | 2.8% |

Our evaluation uses the complete KITTI 2012 (194 pairs) and KITTI 2015 (200 pairs) training sets, so values may differ slightly from the paper because of sparse ground truth, averaging, image handling, and evaluation split details. The fine-tuned models should be compared against the zero-shot results printed above using the same splits and metrics. Lower EPE and D1 are better.

In [ ]:
# Run Protocol A: KITTI 2012 -> KITTI 2015 validation/test
protocol_a_path = OUTPUT / "protocol_a_finetuned.pth"
model_a = train_model(protocol_a["train"], protocol_a["val"], protocol_a_path, steps=5000, lr=1e-5, seed=2025)

zero_model_a = make_model()
zero_a_val = kitti_metrics(zero_model_a, protocol_a["val"])
zero_a_test = kitti_metrics(zero_model_a, protocol_a["test"])
ft_a_val = kitti_metrics(model_a, protocol_a["val"])
ft_a_test = kitti_metrics(model_a, protocol_a["test"])
del zero_model_a
summarize(zero_a_val, "Protocol A zero-shot validation")
summarize(ft_a_val, "Protocol A fine-tuned validation")
summarize(zero_a_test, "Protocol A zero-shot held-out test")
summarize(ft_a_test, "Protocol A fine-tuned held-out test")
del model_a
torch.cuda.empty_cache()


In [ ]:
# Complete Protocol A before starting Protocol B: Middlebury retention
MIDDLEBURY = DATA / "middlebury"
download("https://vision.middlebury.edu/stereo/submit3/zip/MiddEval3-data-H.zip", MIDDLEBURY / "data-H.zip")
download("https://vision.middlebury.edu/stereo/submit3/zip/MiddEval3-GT0-H.zip", MIDDLEBURY / "gt0-H.zip")
extract_once(MIDDLEBURY / "data-H.zip", MIDDLEBURY / "data-H")
extract_once(MIDDLEBURY / "gt0-H.zip", MIDDLEBURY / "gt0-H")
MIDDLEBURY_DATA = next(MIDDLEBURY.rglob("data-H/MiddEval3"))
MIDDLEBURY_GT = next(MIDDLEBURY.rglob("gt0-H/MiddEval3"))

def read_pfm(path):
    with open(path, "rb") as file:
        if file.readline().strip() != b"Pf":
            raise ValueError(path)
        width, height = map(int, file.readline().split())
        scale = float(file.readline())
        endian = "<" if scale < 0 else ">"
        values = np.fromfile(file, endian + "f4")
    return np.flipud(values.reshape(height, width)).astype(np.float32)

def middlebury_metrics(model, label):
    model.eval()
    root = MIDDLEBURY_DATA / "trainingH"
    gt_root = MIDDLEBURY_GT / "trainingH"
    scenes = sorted(p for p in root.iterdir() if p.is_dir())
    rows = []
    print(f"Starting {label} Middlebury inference on {len(scenes)} scenes...", flush=True)
    for scene_number, scene in enumerate(scenes, start=1):
        print(f"Middlebury inference {scene_number}/{len(scenes)} ({scene.name})", flush=True)
        left = imageio.imread(scene / "im0.png")
        right = imageio.imread(scene / "im1.png")
        gt = read_pfm(gt_root / scene.name / "disp0GT.pfm")
        mask = imageio.imread(gt_root / scene.name / "mask0nocc.png") == 255
        h, w = left.shape[:2]
        x0 = torch.from_numpy(left).cuda().float().permute(2, 0, 1)[None]
        x1 = torch.from_numpy(right).cuda().float().permute(2, 0, 1)[None]
        padder = InputPadder(x0.shape, divis_by=32, force_square=False)
        x0, x1 = padder.pad(x0, x1)
        with torch.no_grad(), torch.cuda.amp.autocast(True):
            prediction = model.forward(x0, x1, iters=32, test_mode=True)
        prediction = padder.unpad(prediction.float()).cpu().numpy().reshape(h, w)
        valid = np.isfinite(prediction) & np.isfinite(gt) & mask
        error = np.abs(prediction[valid] - gt[valid])
        rows.append({"scene": scene.name, "valid_pixels": int(valid.sum()), "bad_pixels": int((error > 2).sum()), "epe": float(error.mean())})
    total_valid = sum(row["valid_pixels"] for row in rows)
    print(f"{label} Middlebury weighted EPE: {sum(row['epe'] * row['valid_pixels'] for row in rows) / total_valid:.4f}")
    print(f"{label} Middlebury weighted BP-2: {100 * sum(row['bad_pixels'] for row in rows) / total_valid:.4f}%")
    return rows

def load_finetuned(path):
    model = make_model()
    state = torch.load(path, map_location="cpu", weights_only=False)
    model.load_state_dict(state["model"])
    model.eval()
    set_backbone_eval(model)
    return model

if protocol_a_path.exists() and "middlebury_a" not in globals():
    model_a_middlebury = load_finetuned(protocol_a_path)
    middlebury_a = middlebury_metrics(model_a_middlebury, "Protocol A fine-tuned")
    del model_a_middlebury
else:
    middlebury_a = None
    print("Protocol A checkpoint not found; skipping Protocol A Middlebury evaluation.")
torch.cuda.empty_cache()


In [ ]:
# Run Protocol B: KITTI 2012 train/validation -> all KITTI 2015
protocol_b_path = OUTPUT / "protocol_b_finetuned.pth"
model_b = train_model(protocol_b["train"], protocol_b["val"], protocol_b_path, steps=5000, lr=1e-5, seed=2026)

zero_model_b = make_model()
zero_b_val = kitti_metrics(zero_model_b, protocol_b["val"])
zero_b_test = kitti_metrics(zero_model_b, protocol_b["test"])
ft_b_val = kitti_metrics(model_b, protocol_b["val"])
ft_b_test = kitti_metrics(model_b, protocol_b["test"])
del zero_model_b
summarize(zero_b_val, "Protocol B zero-shot validation")
summarize(ft_b_val, "Protocol B fine-tuned validation")
summarize(zero_b_test, "Protocol B zero-shot KITTI 2015")
summarize(ft_b_test, "Protocol B fine-tuned KITTI 2015")
del model_b
torch.cuda.empty_cache()


## Middlebury retention check

The next cell downloads the half-resolution Middlebury input and public left-view GT, then evaluates the zero-shot and both KITTI-adapted models. It uses `mask == 255` because Middlebury's non-occlusion mask stores 255 for non-occluded pixels and 128 for occluded pixels.


In [ ]:
# Download Middlebury half-resolution data for retention evaluation
MIDDLEBURY = DATA / "middlebury"
download("https://vision.middlebury.edu/stereo/submit3/zip/MiddEval3-data-H.zip", MIDDLEBURY / "data-H.zip")
download("https://vision.middlebury.edu/stereo/submit3/zip/MiddEval3-GT0-H.zip", MIDDLEBURY / "gt0-H.zip")
extract_once(MIDDLEBURY / "data-H.zip", MIDDLEBURY / "data-H")
extract_once(MIDDLEBURY / "gt0-H.zip", MIDDLEBURY / "gt0-H")
MIDDLEBURY_DATA = next(MIDDLEBURY.rglob("data-H/MiddEval3"))
MIDDLEBURY_GT = next(MIDDLEBURY.rglob("gt0-H/MiddEval3"))

def read_pfm(path):
    with open(path, "rb") as file:
        if file.readline().strip() != b"Pf":
            raise ValueError(path)
        width, height = map(int, file.readline().split())
        scale = float(file.readline())
        endian = "<" if scale < 0 else ">"
        values = np.fromfile(file, endian + "f4")
    return np.flipud(values.reshape(height, width)).astype(np.float32)

def middlebury_metrics(model):
    model.eval()
    rows = []
    root = MIDDLEBURY_DATA / "trainingH"
    gt_root = MIDDLEBURY_GT / "trainingH"
    scenes = sorted(p for p in root.iterdir() if p.is_dir())
    print(f"Starting Middlebury inference on {len(scenes)} scenes...", flush=True)
    for scene_number, scene in enumerate(scenes, start=1):
        print(f"Middlebury inference {scene_number}/{len(scenes)} ({scene.name})", flush=True)
        left = imageio.imread(scene / "im0.png")
        right = imageio.imread(scene / "im1.png")
        gt = read_pfm(gt_root / scene.name / "disp0GT.pfm")
        mask = imageio.imread(gt_root / scene.name / "mask0nocc.png") == 255
        h, w = left.shape[:2]
        x0 = torch.from_numpy(left).cuda().float().permute(2, 0, 1)[None]
        x1 = torch.from_numpy(right).cuda().float().permute(2, 0, 1)[None]
        padder = InputPadder(x0.shape, divis_by=32, force_square=False)
        x0, x1 = padder.pad(x0, x1)
        with torch.no_grad(), torch.cuda.amp.autocast(True):
            prediction = model.forward(x0, x1, iters=32, test_mode=True)
        prediction = padder.unpad(prediction.float()).cpu().numpy().reshape(h, w)
        valid = np.isfinite(prediction) & np.isfinite(gt) & mask
        error = np.abs(prediction[valid] - gt[valid])
        bad = error > 2
        rows.append({
            "scene": scene.name,
            "valid_pixels": int(valid.sum()),
            "bad_pixels": int(bad.sum()),
            "epe": float(error.mean()),
            "bp2": float(bad.mean() * 100),
        })
    total_valid = sum(row["valid_pixels"] for row in rows)
    total_bad = sum(row["bad_pixels"] for row in rows)
    weighted_epe = sum(
        row["epe"] * row["valid_pixels"] for row in rows
    ) / total_valid
    weighted_bp2 = 100.0 * total_bad / total_valid
    print("Middlebury non-occluded weighted EPE:", weighted_epe)
    print("Middlebury non-occluded weighted BP-2:", weighted_bp2)
    return rows

def load_finetuned(path):
    model = make_model()
    state = torch.load(path, map_location="cpu", weights_only=False)
    model.load_state_dict(state["model"])
    model.eval()
    set_backbone_eval(model)
    return model

zero_model = make_model()
middlebury_zero_shot = middlebury_metrics(zero_model)
del zero_model
torch.cuda.empty_cache()

if protocol_a_path.exists() and "middlebury_a" not in globals():
    model_a = load_finetuned(protocol_a_path)
    middlebury_a_final = middlebury_metrics(model_a)
    del model_a
elif not protocol_a_path.exists():
    print("Protocol A checkpoint not found; skipping Protocol A retention check.")
torch.cuda.empty_cache()

if protocol_b_path.exists():
    model_b = load_finetuned(protocol_b_path)
    middlebury_b_final = middlebury_metrics(model_b)
    del model_b
else:
    print("Protocol B checkpoint not found; skipping Protocol B retention check.")
torch.cuda.empty_cache()


In [ ]:
# Save compact summaries and fine-tuned checkpoints
import json

summary = {}
if all(name in globals() for name in ["zero_a_val", "ft_a_val", "zero_a_test", "ft_a_test"]):
    summary["protocol_a"] = {
        "zero_shot_val": zero_a_val,
        "fine_tuned_val": ft_a_val,
        "zero_shot_test": zero_a_test,
        "fine_tuned_test": ft_a_test,
    }
if all(name in globals() for name in ["zero_b_val", "ft_b_val", "zero_b_test", "ft_b_test"]):
    summary["protocol_b"] = {
        "zero_shot_val": zero_b_val,
        "fine_tuned_val": ft_b_val,
        "zero_shot_test": zero_b_test,
        "fine_tuned_test": ft_b_test,
    }
if "middlebury_zero_shot" in globals():
    summary["middlebury_zero_shot"] = middlebury_zero_shot
if "middlebury_a_final" in globals():
    summary["middlebury_protocol_a"] = middlebury_a_final
elif "middlebury_a" in globals():
    summary["middlebury_protocol_a"] = middlebury_a
if "middlebury_b_final" in globals():
    summary["middlebury_protocol_b"] = middlebury_b_final
(OUTPUT / "kitti_experiment_summary.json").write_text(json.dumps(summary, indent=2))
archive_path = shutil.make_archive("/content/foundationstereo_kitti_outputs", "zip", OUTPUT)
print("Saved:", archive_path)
